This loads the summary statistics file from a study into a pandas dataframe and inspects the dimensions and first few rows.

In [ ]:
import requests
study_id = "STUDY_ID" # enter study id here
# sends a POST request to the Beiwe server to get participant table data
response = requests.post( 
    "https://studies.beiwe.org/get-summary-statistics/v1",
        data={
            "access_key": "BEIWE_ACCESS_KEY", # enter access key here
            "secret_key": "BEIWE_SECRET_KEY", # enter secret key here
            "study_id": "STUDY_ID",
    },
    allow_redirects=False,
)
# if the request succeeds, save the returned CSV to disk
if response.status_code == 200:
    summary_statistics_table = f"summary_statistics_table_{study_id}.json"
    print(f"The data has been written to {summary_statistics_table}.")
    with open(summary_statistics_table, "wb") as f:
        f.write(response.content)
# otherwise, print the error returned by the server
else:
    print(response.content)
    print(f"There was an error downloading the data; The request returned an HTTP error of {response.status_code}")

In [ ]:
import pandas as pd
with open(summary_statistics_table, "rb") as f:
    df = pd.read_json(f)
    # this will not work unless the summary_statistics_table variable in cell above has been populated and the file exists

# Renaming columns because of different naming scheme in Forest tree vs. CSV exports.

API_COLUMN_RENAME = {
    "date": "Date",
    "participant_id": "Participant Id",
    "study_id": "Study Id",
    "timezone": "Timezone",
    "beiwe_accelerometer_bytes": "Accelerometer Bytes",
    "beiwe_app_log_bytes": "App Log Bytes",
    "beiwe_bluetooth_bytes": "Bluetooth Bytes",
    "beiwe_calls_bytes": "Calls Bytes",
    "beiwe_devicemotion_bytes": "Devicemotion Bytes",
    "beiwe_gps_bytes": "Gps Bytes",
    "beiwe_gyro_bytes": "Gyro Bytes",
    "beiwe_identifiers_bytes": "Identifiers Bytes",
    "beiwe_ios_log_bytes": "Ios Log Bytes",
    "beiwe_magnetometer_bytes": "Magnetometer Bytes",
    "beiwe_power_state_bytes": "Power State Bytes",
    "beiwe_proximity_bytes": "Proximity Bytes",
    "beiwe_reachability_bytes": "Reachability Bytes",
    "beiwe_survey_answers_bytes": "Survey Answers Bytes",
    "beiwe_survey_timings_bytes": "Survey Timings Bytes",
    "beiwe_texts_bytes": "Texts Bytes",
    "beiwe_audio_recordings_bytes": "Audio Recordings Bytes",
    "beiwe_wifi_bytes": "Wifi Bytes",
    "jasmine_distance_diameter": "Distance Diameter",
    "jasmine_distance_from_home": "Distance From Home",
    "jasmine_distance_traveled": "Distance Traveled",
    "jasmine_flight_distance_average": "Flight Distance Average",
    "jasmine_flight_distance_stddev": "Flight Distance Stddev",
    "jasmine_flight_duration_average": "Flight Duration Average",
    "jasmine_flight_duration_stddev": "Flight Duration Stddev",
    "jasmine_home_duration": "Home Duration",
    "jasmine_gyration_radius": "Gyration Radius",
    "jasmine_significant_location_count": "Significant Location Count",
    "jasmine_significant_location_entropy": "Significant Location Entropy",
    "jasmine_pause_time": "Pause Time",
    "jasmine_obs_duration": "Obs Duration",
    "jasmine_obs_day": "Obs Day",
    "jasmine_obs_night": "Obs Night",
    "jasmine_total_flight_time": "Total Flight Time",
    "jasmine_av_pause_duration": "Av Pause Duration",
    "jasmine_sd_pause_duration": "Sd Pause Duration",
    "jasmine_physical_circadian_rhythm": "Physical Circadian Rhythm",
    "jasmine_physical_circadian_rhythm_stratified": "Physical Circadian Rhythm Stratified",
    "willow_incoming_text_count": "Incoming Text Count",
    "willow_incoming_text_degree": "Incoming Text Degree",
    "willow_incoming_text_length": "Incoming Text Length",
    "willow_outgoing_text_count": "Outgoing Text Count",
    "willow_outgoing_text_degree": "Outgoing Text Degree",
    "willow_outgoing_text_length": "Outgoing Text Length",
    "willow_incoming_text_reciprocity": "Incoming Text Reciprocity",
    "willow_outgoing_text_reciprocity": "Outgoing Text Reciprocity",
    "willow_outgoing_MMS_count": "Outgoing Mms Count",
    "willow_incoming_MMS_count": "Incoming Mms Count",
    "willow_mean_responsiveness_text": "Mean Responsiveness Text",
    "willow_incoming_call_count": "Incoming Call Count",
    "willow_incoming_call_degree": "Incoming Call Degree",
    "willow_incoming_call_duration": "Incoming Call Duration",
    "willow_outgoing_call_count": "Outgoing Call Count",
    "willow_outgoing_call_degree": "Outgoing Call Degree",
    "willow_outgoing_call_duration": "Outgoing Call Duration",
    "willow_missed_call_count": "Missed Call Count",
    "willow_missed_callers": "Missed Callers",
    "willow_mean_responsiveness_call": "Mean Responsiveness Call",
    "willow_call_reciprocity": "Call Reciprocity",
    "willow_uniq_individual_call_or_text_count": "Uniq Individual Call Or Text Count",
    "oak_walking_time": "Walking Time",
    "oak_steps": "Steps",
    "oak_cadence": "Cadence",
}

df = df.rename(columns=API_COLUMN_RENAME)

The below chunk loads CSV file locally rather than API pull - DON'T RUN

In [ ]:
import pandas as pd
# Update this path to whichever CSV you want to validate.
INPUT_CSV = "ENTER_CSV_PATH_HERE"
df = pd.read_csv(INPUT_CSV, keep_default_na=False, na_values=[""])
all_reports = []
print(df.shape)
print(df.head())


In [ ]:
print(df.shape)
print(df.head())

This defines the mapping between each raw Beiwe data stream and the summary statistics that should be generated from that stream. The mappings are used to validate whether summary metrics are present when corresponding raw data exists.

In [ ]:
# Observation-time fields are excluded because they may be generated independently from the core GPS movement metrics.
all_reports = []

STREAMS = {
    "GPS": {
    "raw_col": "Gps Bytes",
    "metric_cols": [
        "Distance Diameter",
        "Distance From Home",
        "Distance Traveled",
        "Flight Distance Average",
        "Flight Distance Stddev",
        "Flight Duration Average",
        "Flight Duration Stddev",
        "Home Duration",
        "Gyration Radius",
        "Significant Location Count",
        "Significant Location Entropy",
        "Total Flight Time",
    ],
},
    "Accelerometer": {
        "raw_col": "Accelerometer Bytes",
        "metric_cols": ["Walking Time", "Steps", "Cadence"],
    },
    "Calls": {
        "raw_col": "Calls Bytes",
        "metric_cols": [
            "Incoming Call Count",
            "Incoming Call Degree",
            "Incoming Call Duration",
            "Outgoing Call Count",
            "Outgoing Call Degree",
            "Outgoing Call Duration",
            "Missed Call Count",
            "Missed Callers",
        ],
    },
    "Texts": {
        "raw_col": "Texts Bytes",
        "metric_cols": [
            "Incoming Text Count",
            "Incoming Text Degree",
            "Incoming Text Length",
            "Outgoing Text Count",
            "Outgoing Text Degree",
            "Outgoing Text Length",
            "Incoming Text Reciprocity",
            "Outgoing Text Reciprocity",
            "Outgoing Mms Count",
            "Incoming Mms Count",
        ],
    },
}

This creates a validation function that compares raw data availability against summary statistic availability for a given data stream. Specifically, this function identifies days with raw data present (raw_col > 0) and checks if all expected summary metrics are available. Then, it returns a report containing the participant ID, date, stream name, and issue type if there are any flags.

In [ ]:
def check_stream(df, stream_name, raw_col, metric_cols):
    raw_present = df[raw_col].fillna(0) > 0
# A metric of 0 is considered present. Only a blank or NaN value is considered missing. 
    missing_metric_count = df[metric_cols].isna().sum(axis=1)
    all_metrics_missing = missing_metric_count == len(metric_cols)
    some_metrics_missing = (missing_metric_count > 0) & ~all_metrics_missing
    metrics_present = missing_metric_count == 0

    issue = pd.Series([None] * len(df), index=df.index)

    issue[raw_present & all_metrics_missing] = "Raw data present but ALL metrics missing"
    issue[raw_present & some_metrics_missing] = "Raw data present but SOME metrics missing"
    issue[~raw_present & metrics_present] = "Metrics present but raw data missing"

    report = df.loc[issue.notna(), ["Participant Id", "Date"]].copy()
    report["Stream"] = stream_name
    report["Issue"] = issue[issue.notna()].values

    return report

Expected Columns: This ensures that the expected columns are present in the data. 

In [ ]:
expected_columns = [
    "Date",
    "Participant Id",
    "Study Id",
    "Timezone",
    "Accelerometer Bytes",
    "App Log Bytes",
    "Bluetooth Bytes",
    "Calls Bytes",
    "Devicemotion Bytes",
    "Gps Bytes",
    "Gyro Bytes",
    "Identifiers Bytes",
    "Ios Log Bytes",
    "Magnetometer Bytes",
    "Power State Bytes",
    "Proximity Bytes",
    "Reachability Bytes",
    "Survey Answers Bytes",
    "Survey Timings Bytes",
    "Texts Bytes",
    "Audio Recordings Bytes",
    "Wifi Bytes",
    "Distance Diameter",
    "Distance From Home",
    "Distance Traveled",
    "Flight Distance Average",
    "Flight Distance Stddev",
    "Flight Duration Average",
    "Flight Duration Stddev",
    "Home Duration",
    "Gyration Radius",
    "Significant Location Count",
    "Significant Location Entropy",
    "Pause Time",
    "Obs Duration",
    "Obs Day",
    "Obs Night",
    "Total Flight Time",
    "Av Pause Duration",
    "Sd Pause Duration",
    "Physical Circadian Rhythm",
    "Physical Circadian Rhythm Stratified",
    "Incoming Text Count",
    "Incoming Text Degree",
    "Incoming Text Length",
    "Outgoing Text Count",
    "Outgoing Text Degree",
    "Outgoing Text Length",
    "Incoming Text Reciprocity",
    "Outgoing Text Reciprocity",
    "Outgoing Mms Count",
    "Incoming Mms Count",
    "Mean Responsiveness Text",
    "Incoming Call Count",
    "Incoming Call Degree",
    "Incoming Call Duration",
    "Outgoing Call Count",
    "Outgoing Call Degree",
    "Outgoing Call Duration",
    "Missed Call Count",
    "Missed Callers",
    "Mean Responsiveness Call",
    "Call Reciprocity",
    "Uniq Individual Call Or Text Count",
    "Walking Time",
    "Steps",
    "Cadence"
]

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

extra_columns = [
    col for col in df.columns
    if col not in expected_columns
]

print("===== Column Schema Check =====")

schema_issues = []

for col in missing_columns:
    schema_issues.append({
        "Participant Id": pd.NA,
        "Date": pd.NA,
        "Check": "Expected Columns",
        "Stream": pd.NA,
        "Column": col,
        "Issue": "Expected column is missing"
    })

for col in extra_columns:
    schema_issues.append({
        "Participant Id": pd.NA,
        "Date": pd.NA,
        "Check": "Expected Columns",
        "Stream": pd.NA,
        "Column": col,
        "Issue": "Unexpected column is present"
    })

if not schema_issues:
    print("PASS: All expected columns are present.")
else:
    schema_report = pd.DataFrame(schema_issues)
    all_reports.append(schema_report)

    print(f"FAIL: {len(schema_report)} schema issue(s) found.")
    display(schema_report)

Duplicate Participant-Days: This checks whether the same Participant Id + Date combination appears more than once in the export. A duplicate would cause every other row-level check below to count that row's issues twice, so this runs before any of them.

In [ ]:
print("===== Duplicate Participant-Days Check =====")

duplicate_counts = (
    df.groupby(["Participant Id", "Date"])
    .size()
    .reset_index(name="Count")
)
duplicate_counts = duplicate_counts[duplicate_counts["Count"] > 1]

if duplicate_counts.empty:
    print("PASS: No duplicate participant-day rows found.")
else:
    duplicate_report = duplicate_counts[["Participant Id", "Date"]].copy()
    duplicate_report["Check"] = "Duplicate Participant-Days"
    duplicate_report["Stream"] = pd.NA
    duplicate_report["Column"] = pd.NA
    duplicate_report["Issue"] = (
        "Appears " + duplicate_counts["Count"].astype(str) + " times in the export"
    )

    all_reports.append(duplicate_report)

    print(
        f"FAIL: {len(duplicate_report)} participant-day combination(s) "
        "are duplicated in the export."
    )
    display(duplicate_report)

Date Validity and Calendar Gaps: This checks two things. First, whether every Date value can actually be parsed as a real date. Second, for each participant, whether every calendar day between their earliest and latest row is present — a missing day in the middle would mean a row silently dropped out of the export.

In [ ]:
print("===== Date Validity Check =====")

parsed_dates = pd.to_datetime(df["Date"], errors="coerce")
unparseable_mask = parsed_dates.isna() & df["Date"].notna()

if not unparseable_mask.any():
    print("PASS: All dates parsed successfully.")
else:
    unparseable_report = df.loc[unparseable_mask, ["Participant Id", "Date"]].copy()
    unparseable_report["Check"] = "Date Validity"
    unparseable_report["Stream"] = pd.NA
    unparseable_report["Column"] = "Date"
    unparseable_report["Issue"] = "Date could not be parsed"

    all_reports.append(unparseable_report)

    print(f"FAIL: {len(unparseable_report)} row(s) have an unparseable Date.")
    display(unparseable_report)

print("\n===== Calendar Gap Check =====")

# Only look at rows with a parseable date -- an unparseable one is already
# caught above and would break the day-range math below.
dated = df.loc[parsed_dates.notna()].copy()
dated["Parsed Date"] = parsed_dates.loc[parsed_dates.notna()]

gap_reports = []

for participant_id, group in dated.groupby("Participant Id"):
    observed_days = pd.to_datetime(group["Parsed Date"].unique())
    full_range = pd.date_range(observed_days.min(), observed_days.max(), freq="D")
    missing_days = full_range.difference(observed_days)

    if len(missing_days) > 0:
        gap_reports.append(pd.DataFrame({
            "Participant Id": participant_id,
            "Date": missing_days,
            "Check": "Calendar Gap",
            "Stream": pd.NA,
            "Column": pd.NA,
            "Issue": "No row exists for this date within the participant's collection range"
        }))

if not gap_reports:
    print("PASS: No calendar gaps found within any participant's date range.")
else:
    gap_report = pd.concat(gap_reports, ignore_index=True)
    all_reports.append(gap_report)

    print(f"FAIL: {len(gap_report)} missing calendar day(s) found across participants.")
    display(gap_report)

Data Type Validation - Numeric: This ensures that columns that are expected to have numeric data are indeed numeric.

In [ ]:
# Build the numeric column list from the defined stream mappings.

stream_numeric_columns = []

for config in STREAMS.values():
    stream_numeric_columns.append(config["raw_col"])
    stream_numeric_columns.extend(config["metric_cols"])

additional_numeric_columns = [
    "Pause Time",
    "Obs Duration",
    "Obs Day",
    "Obs Night",
    "Av Pause Duration",
    "Sd Pause Duration",
    "Physical Circadian Rhythm",
    "Physical Circadian Rhythm Stratified",
    "Mean Responsiveness Text",
    "Mean Responsiveness Call",
    "Call Reciprocity",
    "Uniq Individual Call Or Text Count"
]

numeric_columns = list(
    dict.fromkeys(
        stream_numeric_columns + additional_numeric_columns
    )
)

numeric_columns = [
    col for col in numeric_columns
    if col in df.columns
]

incorrect_types = []

for col in numeric_columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        incorrect_types.append({
            "Participant Id": pd.NA,
            "Date": pd.NA,
            "Check": "Data Type Validation",
            "Stream": pd.NA,
            "Column": col,
            "Issue": f"Expected numeric data type but found {df[col].dtype}"
        })

print("===== Data Type Validation =====")

if not incorrect_types:
    print("PASS: All expected numeric columns have numeric data types.")
else:
    data_type_report = pd.DataFrame(incorrect_types)
    all_reports.append(data_type_report)

    print(
        f"FAIL: {len(data_type_report)} column(s) "
        "have incorrect data types."
    )
    display(data_type_report)

Non-Negative Values: This ensures that there are no negative values in numeric columns.

In [ ]:
negative_reports = []

print("===== Range Checks: Non-negative Values =====")

for col in numeric_columns:
    values = pd.to_numeric(df[col], errors="coerce")
    negative_mask = values.notna() & (values < 0)

    if negative_mask.any():
        invalid_rows = df.loc[
            negative_mask,
            ["Participant Id", "Date", col]
        ].copy()

        invalid_rows["Check"] = "Range Check"
        invalid_rows["Stream"] = pd.NA
        invalid_rows["Column"] = col
        invalid_rows["Issue"] = (
            f"Negative value found: "
            + invalid_rows[col].astype(str)
        )

        negative_reports.append(
            invalid_rows[
                [
                    "Participant Id",
                    "Date",
                    "Check",
                    "Stream",
                    "Column",
                    "Issue"
                ]
            ]
        )

if not negative_reports:
    print("PASS: No negative values found in numeric metric columns.")
else:
    negative_values_report = pd.concat(
        negative_reports,
        ignore_index=True
    )

    all_reports.append(negative_values_report)

    print(
        f"FAIL: {len(negative_values_report)} "
        "negative value issue(s) found."
    )
    display(negative_values_report)

Physical Bounds: This check identifies duration variables that fall outside their natural range. Daily duration variables must be between 0 and 25 hours.

In [ ]:
duration_bounds = {
    "Home Duration": (0, 25),
    "Obs Day": (0, 25),
    "Obs Night": (0, 25),
    "Obs Duration": (0, 25),
}

physical_bound_reports = []

print("===== Physical Bounds Check =====")

for col, (lower_bound, upper_bound) in duration_bounds.items():
    if col not in df.columns:
        continue

    values = pd.to_numeric(df[col], errors="coerce")

    invalid_mask = (
        values.notna()
        & (
            (values < lower_bound)
            | (values > upper_bound)
        )
    )

    if invalid_mask.any():
        invalid_rows = df.loc[
            invalid_mask,
            ["Participant Id", "Date", col]
        ].copy()

        invalid_rows["Check"] = "Physical Bounds"
        invalid_rows["Stream"] = pd.NA
        invalid_rows["Column"] = col
        invalid_rows["Issue"] = (
            f"{col} must be between "
            f"{lower_bound} and {upper_bound} hours"
        )

        physical_bound_reports.append(
            invalid_rows[
                [
                    "Participant Id",
                    "Date",
                    "Check",
                    "Stream",
                    "Column",
                    "Issue"
                ]
            ]
        )

if not physical_bound_reports:
    print("PASS: All duration variables are within physical bounds.")
else:
    physical_bounds_report = pd.concat(
        physical_bound_reports,
        ignore_index=True
    )

    all_reports.append(physical_bounds_report)

    print(
        f"FAIL: {len(physical_bounds_report)} "
        "physical-bound issue(s) found."
    )
    display(physical_bounds_report)

Observation Duration: This checks that observation day + observation night = observation duration (approximately). 

In [ ]:
required_cols = [
    "Participant Id",
    "Date",
    "Obs Day",
    "Obs Night",
    "Obs Duration"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

print("===== Observation Duration Consistency Check =====")

if missing_cols:
    print(f"SKIP: Missing required columns: {missing_cols}")
else:
    tolerance = 0.01

    invalid_mask = (
        df["Obs Day"].notna()
        & df["Obs Night"].notna()
        & df["Obs Duration"].notna()
        & (
            (
                df["Obs Day"]
                + df["Obs Night"]
                - df["Obs Duration"]
            ).abs() > tolerance
        )
    )

    if not invalid_mask.any():
        print(
            "PASS: Obs Day + Obs Night is approximately "
            "equal to Obs Duration."
        )
    else:
        observation_report = df.loc[
            invalid_mask,
            [
                "Participant Id",
                "Date",
                "Obs Day",
                "Obs Night",
                "Obs Duration"
            ]
        ].copy()

        observation_report["Check"] = (
            "Observation Duration Consistency"
        )
        observation_report["Stream"] = "GPS"
        observation_report["Column"] = "Obs Duration"
        observation_report["Issue"] = (
            "Obs Day + Obs Night does not equal Obs Duration"
        )

        observation_report = observation_report[
            [
                "Participant Id",
                "Date",
                "Check",
                "Stream",
                "Column",
                "Issue"
            ]
        ]

        all_reports.append(observation_report)

        print(
            f"FAIL: {len(observation_report)} "
            "observation-duration issue(s) found."
        )
        display(observation_report)

Call Counts vs. Durations Check: This chunk will flag if the incoming call count = 0 and incoming call duration is > 0 because you cannot accumulate call duration with no calls.

In [ ]:
required_cols = [
    "Participant Id",
    "Date",
    "Incoming Call Count",
    "Incoming Call Duration",
    "Outgoing Call Count",
    "Outgoing Call Duration"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

print("===== Call Counts vs. Durations Check =====")

call_reports = []

if missing_cols:
    print(f"SKIP: Missing required columns: {missing_cols}")
else:
    incoming_mask = (
        (df["Incoming Call Count"] == 0)
        & (df["Incoming Call Duration"] > 0)
    )

    if incoming_mask.any():
        incoming_report = df.loc[
            incoming_mask,
            ["Participant Id", "Date"]
        ].copy()

        incoming_report["Check"] = "Call Counts vs. Durations"
        incoming_report["Stream"] = "Calls"
        incoming_report["Column"] = "Incoming Call Duration"
        incoming_report["Issue"] = (
            "Incoming Call Count is 0 but "
            "Incoming Call Duration is greater than 0"
        )

        call_reports.append(incoming_report)

    outgoing_mask = (
        (df["Outgoing Call Count"] == 0)
        & (df["Outgoing Call Duration"] > 0)
    )

    if outgoing_mask.any():
        outgoing_report = df.loc[
            outgoing_mask,
            ["Participant Id", "Date"]
        ].copy()

        outgoing_report["Check"] = "Call Counts vs. Durations"
        outgoing_report["Stream"] = "Calls"
        outgoing_report["Column"] = "Outgoing Call Duration"
        outgoing_report["Issue"] = (
            "Outgoing Call Count is 0 but "
            "Outgoing Call Duration is greater than 0"
        )

        call_reports.append(outgoing_report)

    if not call_reports:
        print("PASS: No call count/duration inconsistencies found.")
    else:
        call_consistency_report = pd.concat(
            call_reports,
            ignore_index=True
        )

        all_reports.append(call_consistency_report)

        print(
            f"FAIL: {len(call_consistency_report)} "
            "call consistency issue(s) found."
        )
        display(call_consistency_report)

Text Counts vs. Lengths Check: This will flag if the incoming or outgoing text count = 0 and the corresponding text length is > 0.

In [ ]:
required_cols = [
    "Participant Id",
    "Date",
    "Incoming Text Count",
    "Incoming Text Length",
    "Outgoing Text Count",
    "Outgoing Text Length"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

print("===== Text Counts vs. Lengths Check =====")

text_reports = []

if missing_cols:
    print(f"SKIP: Missing required columns: {missing_cols}")
else:
    incoming_mask = (
        (df["Incoming Text Count"] == 0)
        & (df["Incoming Text Length"] > 0)
    )

    if incoming_mask.any():
        incoming_report = df.loc[
            incoming_mask,
            ["Participant Id", "Date"]
        ].copy()

        incoming_report["Check"] = "Text Counts vs. Lengths"
        incoming_report["Stream"] = "Texts"
        incoming_report["Column"] = "Incoming Text Length"
        incoming_report["Issue"] = (
            "Incoming Text Count is 0 but "
            "Incoming Text Length is greater than 0"
        )

        text_reports.append(incoming_report)

    outgoing_mask = (
        (df["Outgoing Text Count"] == 0)
        & (df["Outgoing Text Length"] > 0)
    )

    if outgoing_mask.any():
        outgoing_report = df.loc[
            outgoing_mask,
            ["Participant Id", "Date"]
        ].copy()

        outgoing_report["Check"] = "Text Counts vs. Lengths"
        outgoing_report["Stream"] = "Texts"
        outgoing_report["Column"] = "Outgoing Text Length"
        outgoing_report["Issue"] = (
            "Outgoing Text Count is 0 but "
            "Outgoing Text Length is greater than 0"
        )

        text_reports.append(outgoing_report)

    if not text_reports:
        print("PASS: No text count/length inconsistencies found.")
    else:
        text_consistency_report = pd.concat(
            text_reports,
            ignore_index=True
        )

        all_reports.append(text_consistency_report)

        print(
            f"FAIL: {len(text_consistency_report)} "
            "text consistency issue(s) found."
        )
        display(text_consistency_report)

Degree vs. Count Check: "Degree" is the number of unique people someone called/texted, and "Count" is the total number of calls/texts. Degree can never be higher than Count, since we can't have more unique contacts than total interactions.

In [ ]:
print("===== Degree vs. Count Check =====")

degree_count_pairs = [
    ("Incoming Call Degree", "Incoming Call Count", "Calls"),
    ("Outgoing Call Degree", "Outgoing Call Count", "Calls"),
    ("Incoming Text Degree", "Incoming Text Count", "Texts"),
    ("Outgoing Text Degree", "Outgoing Text Count", "Texts"),
]

degree_reports = []

for degree_col, count_col, stream_name in degree_count_pairs:
    if degree_col not in df.columns or count_col not in df.columns:
        continue

    degree_values = pd.to_numeric(df[degree_col], errors="coerce")
    count_values = pd.to_numeric(df[count_col], errors="coerce")

    invalid_mask = (
        degree_values.notna()
        & count_values.notna()
        & (degree_values > count_values)
    )

    if invalid_mask.any():
        invalid_rows = df.loc[
            invalid_mask,
            ["Participant Id", "Date", degree_col, count_col]
        ].copy()

        invalid_rows["Check"] = "Degree vs. Count"
        invalid_rows["Stream"] = stream_name
        invalid_rows["Column"] = degree_col
        invalid_rows["Issue"] = f"{degree_col} is greater than {count_col}"

        degree_reports.append(
            invalid_rows[
                [
                    "Participant Id",
                    "Date",
                    "Check",
                    "Stream",
                    "Column",
                    "Issue"
                ]
            ]
        )

if not degree_reports:
    print("PASS: No degree-versus-count inconsistencies found.")
else:
    degree_report = pd.concat(degree_reports, ignore_index=True)
    all_reports.append(degree_report)

    print(
        f"FAIL: {len(degree_report)} "
        "degree-versus-count issue(s) found."
    )
    display(degree_report)

Reciprocity Bounds Check: "Call Reciprocity" is a true ratio (1 = perfect reciprocity, 0 = one-directional), so it should always fall between 0 and 1. Note: despite the similar name, "Incoming/Outgoing Text Reciprocity" are NOT ratios - Forest defines them as raw counts ("number of received/sent SMS without response"), so they're intentionally excluded here. They are already covered by the Non-Negative Values check above, which is the only real constraint on a count.

In [ ]:
reciprocity_bounds = {
    "Call Reciprocity": (0, 1),
}

reciprocity_reports = []

print("===== Reciprocity Bounds Check =====")

for col, (lower_bound, upper_bound) in reciprocity_bounds.items():
    if col not in df.columns:
        continue

    values = pd.to_numeric(df[col], errors="coerce")

    invalid_mask = (
        values.notna()
        & (
            (values < lower_bound)
            | (values > upper_bound)
        )
    )

    if invalid_mask.any():
        invalid_rows = df.loc[
            invalid_mask,
            ["Participant Id", "Date", col]
        ].copy()

        invalid_rows["Check"] = "Reciprocity Bounds"
        invalid_rows["Stream"] = pd.NA
        invalid_rows["Column"] = col
        invalid_rows["Issue"] = (
            f"{col} must be between {lower_bound} and {upper_bound}"
        )

        reciprocity_reports.append(
            invalid_rows[
                [
                    "Participant Id",
                    "Date",
                    "Check",
                    "Stream",
                    "Column",
                    "Issue"
                ]
            ]
        )

if not reciprocity_reports:
    print("PASS: All reciprocity values are within 0 and 1.")
else:
    reciprocity_report = pd.concat(
        reciprocity_reports,
        ignore_index=True
    )

    all_reports.append(reciprocity_report)

    print(
        f"FAIL: {len(reciprocity_report)} "
        "reciprocity-bound issue(s) found."
    )
    display(reciprocity_report)

GPS Movement Consistency: This will flag if distance traveled = 0 and flight distance average is > 0.

In [ ]:
required_cols = [
    "Participant Id",
    "Date",
    "Distance Traveled",
    "Flight Distance Average"
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

print("===== GPS Movement Consistency Check =====")

if missing_cols:
    print(f"SKIP: Missing required columns: {missing_cols}")
else:
    invalid_mask = (
        (df["Distance Traveled"] == 0)
        & (df["Flight Distance Average"] > 0)
    )

    if not invalid_mask.any():
        print("PASS: No GPS movement inconsistencies found.")
    else:
        gps_report = df.loc[
            invalid_mask,
            ["Participant Id", "Date"]
        ].copy()

        gps_report["Check"] = "GPS Movement Consistency"
        gps_report["Stream"] = "GPS"
        gps_report["Column"] = "Flight Distance Average"
        gps_report["Issue"] = (
            "Distance Traveled is 0 but "
            "Flight Distance Average is greater than 0"
        )

        all_reports.append(gps_report)

        print(
            f"FAIL: {len(gps_report)} "
            "GPS movement issue(s) found."
        )
        display(gps_report)

Raw Data vs. Summary Metrics Check: For each data stream (GPS, Accelerometer, Calls, Texts), this compares two things for every participant-day: 1. Is there raw data present? 2. Are the summary metrics for that stream all filled in?

In [ ]:
stream_reports = []

for stream_name, config in STREAMS.items():

    # Check that all required columns exist before running the validation
    required_stream_cols = [
        config["raw_col"],
        *config["metric_cols"]
    ]

    missing_stream_cols = [
        col for col in required_stream_cols
        if col not in df.columns
    ]

    if missing_stream_cols:
        print(
            f"SKIP: {stream_name} check skipped because "
            f"required column(s) are missing: {missing_stream_cols}"
        )
        continue

    # Run the validation
    stream_report = check_stream(
        df,
        stream_name,
        config["raw_col"],
        config["metric_cols"]
    )

    if not stream_report.empty:
        stream_report["Check"] = "Raw Data vs. Summary Metrics"
        stream_report["Column"] = pd.NA

        stream_report = stream_report[
            [
                "Participant Id",
                "Date",
                "Check",
                "Stream",
                "Column",
                "Issue"
            ]
        ]

        stream_reports.append(stream_report)

print("===== Raw Data vs. Summary Metrics Check =====")

if stream_reports:
    raw_metrics_report = pd.concat(
        stream_reports,
        ignore_index=True
    )

    all_reports.append(raw_metrics_report)

    # Break the total down by issue type, since "ALL metrics missing" is the
    # genuine processing gap we care about, while "SOME metrics missing" is
    # often expected (e.g. stddev needs 2+ flights) and lower priority.
    issue_counts = raw_metrics_report["Issue"].value_counts()

    print(
        f"FAIL: {len(raw_metrics_report)} "
        "raw-versus-metric mismatch(es) found."
    )
    for issue_label, count in issue_counts.items():
        print(f"  {count} — {issue_label}")

    display(raw_metrics_report)

else:
    print("PASS: No raw-versus-metric mismatches found.")

This saves the final report as a CSV with a complete table of any issues found.

In [ ]:
report_columns = [
    "Participant Id",
    "Date",
    "Check",
    "Stream",
    "Column",
    "Issue"
]

if all_reports:
    final_report = pd.concat(
        all_reports,
        ignore_index=True
    )

    final_report = final_report[report_columns]

    # Guard against duplicate rows from re-running earlier check cells
    # (e.g. re-executing a check cell without restarting the kernel first,
    # which would otherwise append the same issues to all_reports twice)
    before_dedup = len(final_report)
    final_report = final_report.drop_duplicates().reset_index(drop=True)
    duplicates_removed = before_dedup - len(final_report)

else:
    final_report = pd.DataFrame(
        columns=report_columns
    )
    duplicates_removed = 0

print("===== Complete Validation Report =====")
print(f"Total issues found: {len(final_report)}")

if duplicates_removed > 0:
    print(
        f"NOTE: Removed {duplicates_removed} duplicate row(s) — "
        "likely from a check cell being run more than once. "
        "Consider Kernel > Restart & Run All for a clean run."
    )

if final_report.empty:
    print("PASS: No validation issues found.")
else:
    display(final_report)

final_report.to_csv(
    "validation_report.csv",
    index=False
)

print("Saved validation_report.csv")